In [0]:
%run ./engine

In [0]:
%run ./omop_config

In [0]:
cfg = OMOP_CONFIG
PROFILER = cfg.profiler
RARE = cfg.rare
VALPROT_OVERRIDE = cfg.valprot_override
VOLUME = cfg.target["volume"]
REPORT_DIR = f"{VOLUME}/reports"
GRAPH = cfg.tables
DP = cfg.dp
DISABLE_VALPROT_TABLES = cfg.disable_valprot_tables
PK_ID_BASE = pk_id_base(cfg)

In [0]:
import pandas as pd
import numpy as np
import traceback

_log = []
_failures = []
def out(s):
    _log.append(str(s))
def check(name, cond):
    if cond:
        out(f"  PASS  {name}")
    else:
        out(f"  FAIL  {name}")
        _failures.append(name)

In [0]:
# =====================================================================================
# §1 ENGINE MODEL + DERIVATIONS
# =====================================================================================
try:
    out("== config validity ==")
    check("OMOP_CONFIG validates clean ([] errors)", validate_config(cfg) == [])
    check("17 tables (3 reference + 14 core)", len(cfg.tables) == 17)
    check("subject == person", cfg.subject == "person")
    check("privacy_unit defaults to subject", cfg.privacy_unit == "person")

    from copy import deepcopy
    bad = deepcopy(cfg)
    bad.tables["measurement"].context_fk = ("visit_occurrence_id", "NO_SUCH_PARENT")
    errs = validate_config(bad)
    check("validate flags missing context parent", any("NO_SUCH_PARENT" in e for e in errs))
    bad2 = deepcopy(cfg)
    bad2.subject = "ghost"
    check("validate flags missing subject", any("subject" in e for e in validate_config(bad2)))
    bad3 = deepcopy(cfg)
    bad3.tables["person"].group = "weird"
    check("validate flags bad group", any("group 'weird'" in e for e in validate_config(bad3)))
    bad4 = deepcopy(cfg)
    bad4.tables["measurement"].parent_window = {"parent_start": "x"}   # missing keys
    check("validate flags malformed parent_window",
          any("parent_window" in e for e in validate_config(bad4)))

    out("== topo order + id bands ==")
    expected_order = [
        "location", "care_site", "provider", "person",
        "observation_period", "death", "condition_era", "drug_era", "dose_era",
        "observation_person", "visit_occurrence",
        "measurement", "drug_exposure", "condition_occurrence",
        "procedure_occurrence", "device_exposure", "observation_visit",
    ]
    check("topo_order reproduces pinned legacy order", topo_order(cfg) == expected_order)
    core = topo_order(cfg, "core")
    check("topo_order core filter = 14 tables", len(core) == 14)
    check("topo_order core excludes reference",
          not ({"location", "care_site", "provider"} & set(core)))
    ref = topo_order(cfg, "reference")
    check("topo_order reference filter = 3", ref == ["location", "care_site", "provider"])
    pos = {t: i for i, t in enumerate(topo_order(cfg))}
    ok_parents = True
    for t, s in cfg.tables.items():
        if s.context_fk and pos[s.context_fk[1]] >= pos[t]:
            ok_parents = False
        for p in s.internal_ref_parent.values():
            if pos[p] >= pos[t]:
                ok_parents = False
    check("every parent precedes its child in topo order", ok_parents)

    bands = pk_id_base(cfg)
    check("pk_id_base one band per table", set(bands) == set(cfg.tables))
    check("pk_id_base bands disjoint by 100M",
          bands["location"] == 100_000_000 and bands["observation_visit"] == 17 * 100_000_000)
    check("observation slices land in different bands",
          bands["observation_person"] != bands["observation_visit"])

    out("== encoding_for ==")
    check("concept_id -> categorical",
          encoding_for(cfg, "measurement", "measurement_concept_id", None) == "TABULAR_CATEGORICAL")
    check("source_value -> categorical",
          encoding_for(cfg, "measurement", "measurement_source_value", None) == "TABULAR_CATEGORICAL")
    check("declared freetext_topk col -> categorical",
          encoding_for(cfg, "drug_exposure", "sig", None) == "TABULAR_CATEGORICAL")
    check("sdtype datetime -> TABULAR_DATETIME",
          encoding_for(cfg, "measurement", "measurement_date", "datetime") == "TABULAR_DATETIME")
    check("sdtype numerical -> numeric auto",
          encoding_for(cfg, "measurement", "value_as_number", "numerical") == "TABULAR_NUMERIC_AUTO")
    check("unknown col + no sdtype -> AUTO",
          encoding_for(cfg, "measurement", "mystery_col", None) == "AUTO")
    # R4a: PII provider identifiers are NOT freetext_topk -> encoding_for gives AUTO (they are
    # dropped from training by trained_columns AND nulled at write time by the pipeline).
    check("PII provider_name NOT freetext (encoding AUTO, in cfg.pii)",
          encoding_for(cfg, "provider", "provider_name", None) == "AUTO"
          and "provider_name" in cfg.pii.get("provider", []))
    check("PII npi/dea in cfg.pii[provider]",
          {"npi", "dea"} <= set(cfg.pii.get("provider", [])))
    check("PII care_site_name in cfg.pii[care_site]",
          "care_site_name" in cfg.pii.get("care_site", []))
    check("provider identifiers removed from freetext_topk",
          not ({"provider_name", "npi", "dea", "care_site_name"} & set(cfg.freetext_topk)))

    out("== trained_columns / frame_columns ==")
    person_cols = ["person_id", "gender_concept_id", "birth_datetime",
                   "location_id", "provider_id", "care_site_id"]
    ptr = trained_columns(cfg, "person", person_cols)
    check("person drops pk", "person_id" not in ptr)
    check("person drops reference fks", not ({"location_id", "provider_id", "care_site_id"} & set(ptr)))
    check("person keeps demographics", "gender_concept_id" in ptr and "birth_datetime" in ptr)

    # R4a: provider PII columns are dropped from training.
    prov_cols = ["provider_id", "provider_name", "npi", "dea", "specialty_concept_id", "care_site_id"]
    prov_tr = trained_columns(cfg, "provider", prov_cols)
    check("provider drops PII name/npi/dea from training",
          not ({"provider_name", "npi", "dea"} & set(prov_tr)))
    check("provider keeps specialty_concept_id", "specialty_concept_id" in prov_tr)

    vo_cols = ["visit_occurrence_id", "person_id", "visit_concept_id", "provider_id",
               "care_site_id", "preceding_visit_occurrence_id", "visit_start_date"]
    vtr = trained_columns(cfg, "visit_occurrence", vo_cols)
    check("visit drops pk + context fk(person) + self fk + ref fks",
          not ({"visit_occurrence_id", "person_id", "preceding_visit_occurrence_id",
                "provider_id", "care_site_id"} & set(vtr)))
    check("visit keeps concept + date", "visit_concept_id" in vtr and "visit_start_date" in vtr)
    check("visit is person-context (parent == subject -> NO grandparent drop applied)",
          cfg.tables["visit_occurrence"].context_fk == ("person_id", "person"))

    de_cols = ["drug_exposure_id", "person_id", "visit_occurrence_id", "drug_concept_id",
               "provider_id", "visit_detail_id", "drug_exposure_start_date"]
    dtr = trained_columns(cfg, "drug_exposure", de_cols)
    check("drug_exposure drops pk", "drug_exposure_id" not in dtr)
    check("drug_exposure drops context fk visit_occurrence_id", "visit_occurrence_id" not in dtr)
    check("drug_exposure drops person_id (grandchild: parent != subject)", "person_id" not in dtr)
    check("drug_exposure drops ref fk + dead col",
          "provider_id" not in dtr and "visit_detail_id" not in dtr)
    check("drug_exposure keeps concept", "drug_concept_id" in dtr)

    dfr = frame_columns(cfg, "drug_exposure", de_cols)
    check("frame_columns runs (no TypeError on internal trained_columns call)", dfr is not None)
    check("frame carries pk", "drug_exposure_id" in dfr)
    check("frame carries context fk", "visit_occurrence_id" in dfr)
    check("frame excludes grandparent person_id", "person_id" not in dfr)
    check("frame == trained + key cols (set)",
          set(dfr) == set(dtr) | {"drug_exposure_id", "visit_occurrence_id"})

    death_cols = ["person_id", "death_date", "death_type_concept_id"]
    death_fr = frame_columns(cfg, "death", death_cols)
    check("death frame carries context fk person_id", "person_id" in death_fr)
    check("death frame has no spurious pk", "death_id" not in death_fr)

    out("== build_table_config ==")
    class _FakeDF:
        def __init__(self, cols): self.columns = cols
    de_df = _FakeDF(["drug_exposure_id", "visit_occurrence_id", "drug_concept_id",
                     "drug_exposure_start_date"])
    de_sd = {"drug_exposure_start_date": "datetime", "drug_concept_id": "categorical"}
    btc = build_table_config(cfg, "drug_exposure", de_df, de_sd)
    check("btc name + pk", btc["name"] == "drug_exposure" and btc["primary_key"] == "drug_exposure_id")
    check("btc context foreign_key wired",
          btc["foreign_keys"] == [{"column": "visit_occurrence_id",
                                   "referenced_table": "visit_occurrence", "is_context": True}])
    check("btc DP present (core table)", "differential_privacy" in btc["tabular_model_configuration"])
    check("btc value_protection False (drug_exposure in disable list)",
          btc["tabular_model_configuration"].get("value_protection") is False)
    enc_map = {c["name"]: c["model_encoding_type"] for c in btc["columns"]}
    check("btc key cols AUTO", enc_map["drug_exposure_id"] == "AUTO" and enc_map["visit_occurrence_id"] == "AUTO")
    check("btc concept col categorical", enc_map["drug_concept_id"] == "TABULAR_CATEGORICAL")
    check("btc date col datetime", enc_map["drug_exposure_start_date"] == "TABULAR_DATETIME")

    loc_df = _FakeDF(["location_id", "city", "state"])
    loc_btc = build_table_config(cfg, "location", loc_df, {})
    check("reference btc has no DP", "differential_privacy" not in loc_btc["tabular_model_configuration"])
    check("reference btc has no context fk", "foreign_keys" not in loc_btc)

    death_df = _FakeDF(["person_id", "death_date"])
    death_btc = build_table_config(cfg, "death", death_df, {"death_date": "datetime"})
    check("death btc no primary_key (pk=None)", "primary_key" not in death_btc)
    check("death btc keeps value_protection (not disabled)",
          death_btc["tabular_model_configuration"].get("value_protection") is None)

    out("== dp_block ==")
    dpb = dp_block(cfg)
    check("dp_block max_epsilon", dpb["max_epsilon"] == 2.0)
    check("dp_block value_protection_epsilon", dpb["value_protection_epsilon"] == 3.0)
    check("dp_block delta + noise + grad",
          dpb["delta"] == 1e-6 and dpb["noise_multiplier"] == 1.5 and dpb["max_grad_norm"] == 1.0)

    out("== sdtypes_table_key ==")
    check("split slice maps to source omop_observation",
          sdtypes_table_key(cfg, "observation_person") == "omop_observation"
          and sdtypes_table_key(cfg, "observation_visit") == "omop_observation")
    check("plain table maps to its own omop_ key",
          sdtypes_table_key(cfg, "measurement") == "omop_measurement")

    out("== dataset_config_hash ==")
    h0 = dataset_config_hash(cfg)
    check("dataset_config_hash 16-hex deterministic",
          isinstance(h0, str) and len(h0) == 16 and dataset_config_hash(cfg) == h0)
    saved = cfg.tables["observation_visit"].where
    cfg.tables["observation_visit"].where = "visit_occurrence_id IS NULL"
    h1 = dataset_config_hash(cfg)
    cfg.tables["observation_visit"].where = saved
    check("dataset_config_hash reflects graph shape (where flip changes it)", h0 != h1)
    check("dataset_config_hash restored after revert", dataset_config_hash(cfg) == h0)
    check("dataset_config_hash differs from the profiler config_hash (two payloads by design)",
          dataset_config_hash(cfg) != config_hash())

    out("== introspect ==")
    de_schema = ["drug_exposure_id", "visit_occurrence_id", "person_id", "drug_concept_id",
                 "drug_source_value", "quantity", "drug_exposure_start_date", "sig"]
    de_sdt = {"quantity": "numerical", "drug_exposure_start_date": "datetime"}
    roles = roles_from_schema(cfg, "drug_exposure", de_schema, de_sdt)
    check("introspect concept -> categorical", roles["drug_concept_id"] == "TABULAR_CATEGORICAL")
    check("introspect source_value -> categorical", roles["drug_source_value"] == "TABULAR_CATEGORICAL")
    check("introspect freetext sig -> categorical", roles["sig"] == "TABULAR_CATEGORICAL")
    check("introspect numerical -> numeric auto", roles["quantity"] == "TABULAR_NUMERIC_AUTO")
    check("introspect datetime -> datetime", roles["drug_exposure_start_date"] == "TABULAR_DATETIME")
    amb = ambiguous_cols(roles)
    check("introspect ids ambiguous (AUTO)", set(amb) >= {"drug_exposure_id", "visit_occurrence_id", "person_id"})
    from copy import deepcopy as _dc
    cfg_ov = _dc(cfg)
    cfg_ov.tables["drug_exposure"].encoding_overrides = {"drug_concept_id": "AUTO"}
    roles_ov = roles_from_schema(cfg_ov, "drug_exposure", de_schema, de_sdt)
    check("introspect encoding_override wins over heuristic", roles_ov["drug_concept_id"] == "AUTO")

except Exception as e:
    out("EXCEPTION: " + repr(e))
    out(traceback.format_exc())
    _failures.append("EXCEPTION_engine_section")

In [0]:
# =====================================================================================
# §2 LIBRARY PURE FUNCTIONS
# =====================================================================================
try:
    out("== rekey ==")
    df = pd.DataFrame({"measurement_id": [9, 9_000_000_001, 7], "v": [1, 2, 3]})
    keyed, mp = rekey(df, "measurement_id", base=1000)
    check("rekey dense sequential", list(keyed["measurement_id"]) == [1000, 1001, 1002])
    check("rekey preserves other cols", list(keyed["v"]) == [1, 2, 3])
    check("rekey mapping shape", list(mp.columns) == ["old_measurement_id", "measurement_id"])
    check("rekey mapping old->new", dict(zip(mp["old_measurement_id"], mp["measurement_id"])) ==
          {9: 1000, 9_000_000_001: 1001, 7: 1002})

    ek, em = rekey(df.iloc[0:0], "measurement_id", base=5)
    check("rekey empty df", len(ek) == 0 and len(em) == 0)

    op_keyed, _ = rekey(pd.DataFrame({"observation_id": [1, 2]}), "observation_id",
                        base=PK_ID_BASE["observation_person"])
    ov_keyed, _ = rekey(pd.DataFrame({"observation_id": [1, 2]}), "observation_id",
                        base=PK_ID_BASE["observation_visit"])
    check("observation slices disjoint ids",
          set(op_keyed["observation_id"]).isdisjoint(set(ov_keyed["observation_id"])))

    parent = pd.DataFrame({"visit_occurrence_id": [100, 200]})
    pk_keyed, pk_map = rekey(parent, "visit_occurrence_id", base=10)
    child = pd.DataFrame({"measurement_id": [1, 2, 3], "visit_occurrence_id": [100, 200, 999]})
    ri = remap_fk(child, "visit_occurrence_id", pk_map, "visit_occurrence_id", how="inner")
    check("remap_fk inner drops orphan", len(ri) == 2 and set(ri["visit_occurrence_id"]) == {10, 11})
    rl = remap_fk(child, "visit_occurrence_id", pk_map, "visit_occurrence_id", how="left")
    check("remap_fk left keeps orphan as NaN", len(rl) == 3 and rl["visit_occurrence_id"].isna().sum() == 1)

    visit_keyed0 = pd.DataFrame({"visit_occurrence_id": [10, 11], "person_id": [501, 502]})
    ev = pd.DataFrame({"measurement_id": [1, 2], "visit_occurrence_id": [10, 11], "person_id": [-1, -1]})
    ev2 = derive_parent_key(ev, visit_keyed0, "visit_occurrence_id", "person_id")
    check("derive_parent_key sets person from visit", list(ev2["person_id"]) == [501, 502])

    out("== temporal ==")
    t = pd.DataFrame({"s": pd.to_datetime(["2020-01-10", "2020-02-01"]),
                      "e": pd.to_datetime(["2020-01-05", "2020-02-10"])})
    t1 = order_pair(t, "s", "e")
    check("order_pair fixes end<start", t1.loc[0, "e"] == t1.loc[0, "s"])
    check("order_pair leaves good row", t1.loc[1, "e"] == pd.Timestamp("2020-02-10"))
    check("order_pair idempotent", order_pair(t1, "s", "e").equals(t1))

    ev = pd.DataFrame({"d": pd.to_datetime(["2019-12-01", "2020-01-15", "2020-03-01"])})
    lo = pd.Series(pd.to_datetime(["2020-01-01"] * 3))
    hi = pd.Series(pd.to_datetime(["2020-01-31"] * 3))
    c = clamp_into_window(ev, ["d"], lo, hi)
    check("clamp below->lo", c.loc[0, "d"] == pd.Timestamp("2020-01-01"))
    check("clamp inside untouched", c.loc[1, "d"] == pd.Timestamp("2020-01-15"))
    check("clamp above->hi", c.loc[2, "d"] == pd.Timestamp("2020-01-31"))
    check("clamp idempotent", clamp_into_window(c, ["d"], lo, hi).equals(c))

    evn = pd.DataFrame({"d": pd.to_datetime(["2019-01-01", "2050-01-01"])})
    lon = pd.Series([pd.NaT, pd.NaT]); hin = pd.Series([pd.NaT, pd.NaT])
    cn = clamp_into_window(evn, ["d"], lon, hin)
    check("clamp NaT window leaves dates untouched",
          cn.loc[0, "d"] == pd.Timestamp("2019-01-01") and cn.loc[1, "d"] == pd.Timestamp("2050-01-01"))

    b = pd.DataFrame({"birth_datetime": pd.to_datetime(["2000-01-01", "1990-01-01"]),
                      "ev": pd.to_datetime(["1999-06-01", "1995-01-01"])})
    b1 = birth_before_events(b, "birth_datetime", ["ev"])
    check("birth_before_events pushes event up", b1.loc[0, "ev"] == pd.Timestamp("2000-01-01"))
    check("birth_before_events leaves valid", b1.loc[1, "ev"] == pd.Timestamp("1995-01-01"))

    d = pd.DataFrame({"death_date": pd.to_datetime(["2020-01-01", "2020-06-01", None]),
                      "last_ev": pd.to_datetime(["2020-03-01", "2020-01-01", "2020-01-01"])})
    mask = death_after_events(d, "death_date", "last_ev")
    check("death before last event -> invalid", bool(mask.iloc[0]) is False)
    check("death after last event -> valid", bool(mask.iloc[1]) is True)
    check("no death -> valid", bool(mask.iloc[2]) is True)

    r = pd.DataFrame({"d": pd.to_datetime(["1899-01-01", "2100-06-01", "2020-01-01"])})
    rc = clamp_valid_range(r, ["d"], lo=BUSINESS_MIN, hi=BUSINESS_MAX)
    check("clamp below business min", rc.loc[0, "d"] == BUSINESS_MIN)
    check("clamp above business max", rc.loc[1, "d"] == BUSINESS_MAX)
    check("clamp in-range untouched", rc.loc[2, "d"] == pd.Timestamp("2020-01-01"))
    check("clamp idempotent", clamp_valid_range(rc, ["d"], lo=BUSINESS_MIN, hi=BUSINESS_MAX).equals(rc))
    check("default bounds = pandas representable",
          PANDAS_MIN == pd.Timestamp.min.ceil("D") and PANDAS_MAX == pd.Timestamp.max.floor("D"))

    v = pd.DataFrame({
        "visit_occurrence_id": [11, 12, 13, 21],
        "person_id": [1, 1, 1, 2],
        "visit_start_date": pd.to_datetime(["2020-03-01", "2020-01-01", "2020-02-01", "2020-05-01"]),
    })
    vc = rebuild_preceding_chain(v, "person_id", "visit_occurrence_id",
                                 "visit_start_date", "preceding_visit_occurrence_id")
    vc_by_id = vc.set_index("visit_occurrence_id")
    check("chain first visit no preceding", pd.isna(vc_by_id.loc[12, "preceding_visit_occurrence_id"]))
    check("chain second points to first", vc_by_id.loc[13, "preceding_visit_occurrence_id"] == 12)
    check("chain third points to second", vc_by_id.loc[11, "preceding_visit_occurrence_id"] == 13)
    check("chain new person resets", pd.isna(vc_by_id.loc[21, "preceding_visit_occurrence_id"]))
    check("chain no self reference",
          not (vc["visit_occurrence_id"] == vc["preceding_visit_occurrence_id"]).any())

    dd = pd.DataFrame({"d_date": pd.to_datetime(["2020-01-01"]),
                       "d_datetime": pd.to_datetime(["2020-05-05 13:00:00"])})
    dd2 = date_from_datetime(dd, [("d_date", "d_datetime")])
    check("date re-derived from datetime", dd2.loc[0, "d_date"] == pd.Timestamp("2020-05-05"))

    out("== fk_remap ==")
    location = pd.DataFrame({"location_id": [1, 2]})
    care_site = pd.DataFrame({"care_site_id": [10, 11], "location_id": [1, 2]})
    provider = pd.DataFrame({"provider_id": [100, 101, 102], "care_site_id": [10, 11, 10]})
    ref_frames = {"provider": provider, "care_site": care_site, "location": location}
    check("chain_keys derives pk order from cfg.reference_chain",
          chain_keys(cfg) == ["provider_id", "care_site_id", "location_id"])
    tuples = build_chain_tuples(cfg, ref_frames)
    check("tuples one row per provider", len(tuples) == 3)
    check("tuples consistent prov->cs->loc",
          set(map(tuple, tuples[["provider_id", "care_site_id", "location_id"]].itertuples(index=False, name=None)))
          == {(100, 10, 1), (101, 11, 2), (102, 10, 1)})

    person = pd.DataFrame({"person_id": [1, 2, 3, 4, 5],
                           "provider_id": [np.nan]*5, "care_site_id": [np.nan]*5, "location_id": [np.nan]*5})
    pa = assign_subject_reference(cfg, person, tuples, seed=42)
    check("person reference assigned (no NaN)",
          pa[["provider_id", "care_site_id", "location_id"]].notna().all().all())
    check("person tuples consistent", tuple_consistency_violations(pa, tuples) == 0)

    visit = pd.DataFrame({"visit_occurrence_id": [11, 12, 13],
                          "person_id": [1, 2, 3], "provider_id": [np.nan]*3, "care_site_id": [np.nan]*3})
    va = assign_child_reference(cfg, visit, "visit_occurrence", pa, tuples, seed=7)
    va_full = va.merge(care_site[["care_site_id", "location_id"]], on="care_site_id", how="left")
    check("visit tuples consistent", tuple_consistency_violations(va_full, tuples) == 0)

    event = pd.DataFrame({"measurement_id": [1, 2, 3], "visit_occurrence_id": [11, 12, 13],
                          "provider_id": [np.nan]*3})
    ei = inherit_parent_col(event, va, "visit_occurrence_id", "provider_id")
    check("event provider inherited from visit",
          list(ei["provider_id"]) == list(va.set_index("visit_occurrence_id").loc[[11,12,13], "provider_id"]))

    vk = pd.DataFrame({"visit_occurrence_id": [11, 12, 13], "person_id": [1, 1, 2]})
    re_event = pd.DataFrame({"drug_exposure_id": [101, 102, 103, 104],
                             "person_id": [1, 1, 2, 9], "drug_concept_id": [1, 2, 3, 4]})
    av = assign_visit_within_person(re_event, vk, seed=3)
    check("assign_visit person1 from own visits", set(av.iloc[0:2]["visit_occurrence_id"]) <= {11, 12})
    check("assign_visit person2 gets only its visit", av.iloc[2]["visit_occurrence_id"] == 13)
    check("assign_visit person-with-no-visit -> NaN", pd.isna(av.iloc[3]["visit_occurrence_id"]))
    vk_owner = vk.set_index("visit_occurrence_id")["person_id"].to_dict()
    consistent = all(vk_owner.get(vv) == pp for vv, pp in
                     zip(av["visit_occurrence_id"], av["person_id"]) if pd.notna(vv))
    check("assign_visit all within own person", consistent)
    check("assign_visit no orphan visit fk",
          orphan_count(av, "visit_occurrence_id", vk, "visit_occurrence_id") == 0)

    big = pd.DataFrame({"visit_occurrence_id": list(range(1000))})
    nb = reinject_null_fk(big, "visit_occurrence_id", null_rate=0.05, seed=1)
    check("reinject ~5% null", abs(nb["visit_occurrence_id"].isna().sum() - 50) <= 1)

    par = pd.DataFrame({"person_id": [1, 2, 3]})
    chi = pd.DataFrame({"person_id": [1, 2, 99, np.nan]})
    check("orphan_count counts only bad non-null", orphan_count(chi, "person_id", par, "person_id") == 1)

    bad_assign = pd.DataFrame({"provider_id": [100], "care_site_id": [11], "location_id": [1]})
    check("tuple violation detected", tuple_consistency_violations(bad_assign, tuples) == 1)

    capdf = pd.DataFrame({
        "drug_exposure_id": list(range(9)),
        "visit_occurrence_id": [1, 1, 1, 1, 1, 2, 2, 3, np.nan],
    })
    capped = cap_per_visit(capdf, "visit_occurrence_id", cap=2, seed=11)
    vc_counts = capped[capped["visit_occurrence_id"].notna()].groupby("visit_occurrence_id").size()
    check("cap_per_visit caps each visit to <= cap", (vc_counts <= 2).all())
    check("cap_per_visit visit1 reduced 5->2", int(vc_counts.get(1, 0)) == 2)
    check("cap_per_visit visit2 kept (2<=2)", int(vc_counts.get(2, 0)) == 2)
    check("cap_per_visit visit3 kept (1<=2)", int(vc_counts.get(3, 0)) == 1)
    check("cap_per_visit keeps NULL-fk rows", int(capped["visit_occurrence_id"].isna().sum()) == 1)
    check("cap_per_visit deterministic given seed",
          cap_per_visit(capdf, "visit_occurrence_id", 2, 11).sort_values("drug_exposure_id")
          ["drug_exposure_id"].tolist() == capped.sort_values("drug_exposure_id")["drug_exposure_id"].tolist())
    check("cap_per_visit no-op when cap>=max group",
          len(cap_per_visit(capdf, "visit_occurrence_id", 99, 11)) == len(capdf))
    check("cap_per_visit empty df ok", len(cap_per_visit(capdf.iloc[0:0], "visit_occurrence_id", 2, 1)) == 0)

    deaths = pd.DataFrame({"person_id": list(range(40)), "death_date": pd.NaT})
    cal = calibrate_rate(deaths, n_parents=100, target_rate=0.10, seed=7, tol=0.02)
    check("calibrate down-samples to target count", len(cal) == 10)
    check("calibrate keeps a subset of original rows",
          set(cal["person_id"]).issubset(set(deaths["person_id"])))
    check("calibrate deterministic given seed",
          calibrate_rate(deaths, 100, 0.10, 7, 0.02)["person_id"].tolist() == cal["person_id"].tolist())
    check("calibrate within tol -> no-op", len(calibrate_rate(deaths, 100, 0.39, 7, 0.02)) == 40)
    check("calibrate below target -> no-op (no invent)", len(calibrate_rate(deaths, 100, 0.80, 7, 0.02)) == 40)
    check("calibrate empty df ok", len(calibrate_rate(deaths.iloc[0:0], 100, 0.10, 7, 0.02)) == 0)
    check("calibrate n_parents=0 -> no-op", len(calibrate_rate(deaths, 0, 0.10, 7, 0.02)) == 40)

    out("== rarity ==")
    check("det_seed deterministic", det_seed(0, "A") == det_seed(0, "A"))
    check("det_seed distinguishes keys", det_seed(0, "A") != det_seed(0, "B"))
    check("det_seed base folded (small base diff distinguishes)", det_seed(0, "A") != det_seed(1, "A"))
    check("det_seed no mod-collision (1000137 vs 1000363)",
          det_seed(300, "1000137") != det_seed(300, "1000363"))
    nsup = noise_support({"1000137": 100, "1000363": 100}, sensitivity=100, eps=1.0, seed_base=300)
    check("noise_support independent for colliding-pair keys",
          det_seed(300, "1000137") != det_seed(300, "1000363") and set(nsup) == {"1000137", "1000363"})

    rparents = pd.DataFrame({"visit_occurrence_id": [1, 2, 3], "person_id": [10, 10, 11]})
    rchild = pd.DataFrame({"visit_occurrence_id": [1, 1, 1, 2], "person_id": [10, 10, 10, 10]})
    s = seqlen_stat(rchild, rparents, "visit_occurrence_id", "person_id",
                    bins=[0, 1, 2, 3, 5, 1000000], parent_cap=50)
    check("seqlen frac_parent_with_child = 2/3", abs(s["frac_parent_with_child"] - 2/3) < 1e-9)
    check("seqlen bin counts sum to n_parents", sum(s["bin_counts"]) == 3)
    check("seqlen no raw max field (removed)", "max_nonzero_len" not in s)
    check("seqlen carries bins", s["bins"] == [0, 1, 2, 3, 5, 1000000])
    check("seqlen bin placement", s["bin_counts"] == [1, 1, 0, 1, 0])
    s_cap = seqlen_stat(rchild, rparents, "visit_occurrence_id", "person_id",
                        bins=[0, 1, 2, 3, 5, 1000000], parent_cap=1)
    check("seqlen parent_cap limits parents per patient", s_cap["n_parents"] == 2)
    rparents2 = pd.DataFrame({"visit_occurrence_id": [1], "person_id": [10]})
    rchild2 = pd.DataFrame({"visit_occurrence_id": [1] * 1_000_005, "person_id": [10] * 1_000_005})
    s2 = seqlen_stat(rchild2, rparents2, "visit_occurrence_id", "person_id",
                     bins=[0, 1, 2, 3, 5, 1000000], parent_cap=50)
    check("seqlen clamps overflow into top bin (no dropped parent)", sum(s2["bin_counts"]) == 1)
    pr_parents = pd.DataFrame({"person_id": [1, 2, 3]})
    pr_child = pd.DataFrame({"person_id": [1, 1, 2]})
    sp = seqlen_stat(pr_child, pr_parents, "person_id", "person_id",
                     bins=[0, 1, 2, 3, 5, 1000000], parent_cap=50)
    check("seqlen person-rooted n_parents = 3", sp["n_parents"] == 3)
    check("seqlen person-rooted frac = 2/3", abs(sp["frac_parent_with_child"] - 2/3) < 1e-9)
    check("seqlen person-rooted bins", sp["bin_counts"] == [1, 1, 1, 0, 0] and sum(sp["bin_counts"]) == 3)

    rpersons = pd.DataFrame({"person_id": [1, 2, 3, 4]})
    rdeaths = pd.DataFrame({"person_id": [2]})
    br = base_rate_stat(rdeaths, rpersons, "person_id")
    check("base_rate raw = 0.25", abs(br["rate"] - 0.25) < 1e-9)
    check("base_rate pos_count = 1", br["pos_count"] == 1)
    check("base_rate n = 4", br["n_parents"] == 4)
    check("base_rate empty parents -> 0", base_rate_stat(rdeaths, rpersons.iloc[0:0], "person_id")["rate"] == 0.0)

    crows = pd.DataFrame({"person_id": [1, 2, 1, 1], "concept_id": ["A", "A", "B", "B"]})
    cs = category_support_stat(crows, "concept_id", "person_id", per_patient_cap=100)
    check("cat A distinct-patient support = 2", cs["support"]["A"] == 2)
    check("cat B distinct-patient support = 1 (not row count 2)", cs["support"]["B"] == 1)
    check("cat cardinality = 2", cs["cardinality"] == 2)
    cs2 = category_support_stat(crows, "concept_id", "person_id", per_patient_cap=1)
    check("cap limits patient to 1 category (A kept)", cs2["support"].get("A") == 2)
    check("cap drops patient1's second category B", "B" not in cs2["support"])
    csn = category_support_stat(pd.DataFrame({"person_id": [1, 2], "concept_id": ["A", None]}),
                                "concept_id", "person_id", 100)
    check("cat drops NULL category", csn["support"] == {"A": 1} and csn["cardinality"] == 1)

    nvals = [noise_count(10, sensitivity=1, eps=1.0, seed=s) for s in range(2000)]
    check("noise_count clamped >= 0", min(nvals) >= 0)
    check("noise_count ~unbiased", abs(np.mean(nvals) - 10) < 0.5)
    check("noise_count deterministic given seed", noise_count(10, 1, 1.0, 7) == noise_count(10, 1, 1.0, 7))
    check("noise_count eps<=0 -> exact rounded", noise_count(10.4, 1, 0.0, 7) == 10)
    rr = noise_rate(0.25, n=1000, sensitivity=1, eps=1.0, seed=7)
    check("noise_rate in [0,1]", 0.0 <= rr <= 1.0)
    check("noise_rate n=0 -> 0", noise_rate(0.5, 0, 1, 1.0, 7) == 0.0)
    nb_ = noise_bins([100, 50, 0], sensitivity=50, eps=1.0, seed=5)
    check("noise_bins length preserved + clamped", len(nb_) == 3 and min(nb_) >= 0)
    check("noise_bins deterministic", noise_bins([100, 50, 0], 50, 1.0, 5) == nb_)
    ns = noise_support({"A": 100, "B": 1}, sensitivity=100, eps=1.0, seed_base=300)
    check("noise_support keys preserved + clamped", set(ns) == {"A", "B"} and min(ns.values()) >= 0)
    check("noise_support deterministic",
          noise_support({"A": 100}, 100, 1.0, 300) == noise_support({"A": 100}, 100, 1.0, 300))

    hdr = profile_header(mode="full", cohort_counts={"person": 30000}, cfg_hash="abc123",
                         seed=42, eps_profiler=1.0, source_version="v1")
    check("header schema_version", hdr["schema_version"] == RARITY_SCHEMA_VERSION)
    check("header carries cfg hash + mode", hdr["config_hash"] == "abc123" and hdr["cohort_mode"] == "full")
    prof = {"header": hdr, "tables": {"death": {"base_rate": {"rate": 0.04}}}}
    check("profile_is_fresh true on match", profile_is_fresh(prof, "abc123", "full") is True)
    check("profile_is_fresh false on hash mismatch", profile_is_fresh(prof, "ZZZ", "full") is False)
    check("profile_is_fresh false on mode mismatch", profile_is_fresh(prof, "abc123", "fixture") is False)
    check("profile_is_fresh false on empty", profile_is_fresh({}, "abc123", "full") is False)

    prof_tables = {
        "x": {"seqlen": {"n_parents": 100, "frac_parent_with_child": 0.02,
                          "bin_counts": [98, 2, 0, 0, 0], "bins": [0, 1, 2, 3, 5, 1000000]}},
        "y": {"seqlen": {"n_parents": 100, "frac_parent_with_child": 0.60,
                          "bin_counts": [40, 60, 0, 0, 0], "bins": [0, 1, 2, 3, 5, 1000000]}},
    }
    px = collapse_policy("x", prof_tables, override={}, legacy_off=[], support_min=0.05)
    check("x collapse-prone -> vp False (2/100<0.05)", px["value_protection"] is False and px["source"] == "profile")
    py = collapse_policy("y", prof_tables, override={}, legacy_off=[], support_min=0.05)
    check("y healthy -> vp True (60/100>=0.05)", py["value_protection"] is True)
    po = collapse_policy("x", prof_tables, override={"x": True}, legacy_off=[], support_min=0.05)
    check("override forces vp True", po["value_protection"] is True and po["source"] == "override")
    pl = collapse_policy("z", {}, override={}, legacy_off=["z"], support_min=0.05)
    check("missing profile + legacy -> vp False", pl["value_protection"] is False and pl["source"] == "legacy_default")
    pm = collapse_policy("w", {}, override={}, legacy_off=["z"], support_min=0.05)
    check("missing profile + not legacy -> safe vp True", pm["value_protection"] is True)

    res = resolve_all_policies(["person", "x", "y", "z"], prof_tables,
                               override={}, legacy_off=["z"], support_min=0.05)
    check("resolve_all on-set = person,y", {t for t in res if res[t]["value_protection"]} == {"person", "y"})

    check("config_hash deterministic 16-hex", isinstance(config_hash(), str) and len(config_hash()) == 16
          and config_hash() == config_hash())
    check("RARITY_CONFIG_HASH is config_hash (identity, shadow-proof)",
          RARITY_CONFIG_HASH is config_hash and RARITY_CONFIG_HASH() == config_hash())
    _h0 = config_hash()
    _spec = GRAPH["observation_visit"]
    _orig_where = _spec.where
    _spec.where = "visit_occurrence_id IS NULL"   # flip the slice predicate
    _h1 = config_hash()
    _spec.where = _orig_where                       # restore
    check("config_hash reflects GRAPH shape (where change flips hash)", _h0 != _h1)
    check("config_hash restored after revert", config_hash() == _h0)
    legacy_on = [t for t in topo_order(cfg, "core") if t not in legacy_valprot_off()]
    ce = composed_epsilon(legacy_on)
    check("composed eps = 28 + 3*n_on + profiler",
          abs(ce["total"] - (DP["max_epsilon"]*len(topo_order(cfg, "core"))
                             + DP["value_protection_epsilon"]*len(legacy_on)
                             + PROFILER["epsilon_profiler"])) < 1e-9)
    check("composed eps profiler term > 0", ce["profiler"] == PROFILER["epsilon_profiler"])
    check("legacy_valprot_off == proven 5",
          set(legacy_valprot_off()) == {"drug_era", "dose_era", "drug_exposure",
                                        "procedure_occurrence", "device_exposure"})
    check("DISABLE_VALPROT_TABLES correct",
          set(DISABLE_VALPROT_TABLES) == {"drug_era", "dose_era", "drug_exposure",
                                          "procedure_occurrence", "device_exposure"})

except Exception as e:
    out("EXCEPTION: " + repr(e))
    out(traceback.format_exc())
    _failures.append("EXCEPTION_library_section")

In [0]:
# Persist log + summary. RAISE on failure so the notebook job status is FAILED.
n_pass = sum(1 for l in _log if l.startswith("  PASS"))
summary = (f"PASS: ALL TESTS PASSED ({n_pass} checks)" if not _failures
           else f"FAIL: {len(_failures)} failure(s): {_failures}")
log_text = "\n".join(_log) + "\n\n" + summary + "\n"
with open(f"{REPORT_DIR}/tests.log", "w") as f:
    f.write(log_text)
print(log_text)
if _failures:
    raise AssertionError(summary + " — see " + f"{REPORT_DIR}/tests.log")
dbutils.notebook.exit(summary)